#### Run common functions notebook

In [0]:
%run ../utils/common_functions

##### Customer

In [0]:
# Fetch the customer table from the bronze schema
customer_df = read_table("customers","bronze").dropDuplicates(["customer_id"])

# Write the customer table to the silver schema
write_table(df = customer_df, table_name = "customers", schema_name = "silver")

##### Geo Location Details

In [0]:
# Fetch the geolocation details table from bronze schema
geo_location_details_df = read_table("geolocation_details","bronze")

# Aggregate the table as it contains multiple entries for a single zip code 
aggregated_geo_location_details_df = geo_location_details_df.groupBy("geolocation_zip_code_prefix") \
                                            .agg(
                                                avg("geolocation_lat").alias("latitude"), 
                                                avg("geolocation_lng").alias("longitude"),
                                                first("geolocation_city").alias("city"),
                                                first("geolocation_state").alias("state_code"))

In [0]:
# Add an additional column for easier identification of Brazil states
state_mapping_df = state_mapping()

# Join the aggregated_geo_location_details_df with the state_mapping_df
aggregated_geo_location_details_df = aggregated_geo_location_details_df.join(broadcast(state_mapping_df), on="state_code", how ="left")

# Write the aggregated_geo_location_details_df to the silver schema
write_table(df = aggregated_geo_location_details_df, table_name = "geolocation_details",schema_name = "silver")

##### Order Items

In [0]:
# Fetch the order items table from bronze schema
order_items_df = read_table("order_items","bronze")

# Add a column to calculate the total va;ue of each order
order_items_df = order_items_df.withColumn("total_value", round(col("price") + col("freight_value"), 2))

# Write the order_items_df to the silver schema
write_table(df = order_items_df, table_name = "order_items", schema_name = "silver")

##### Order Payments

In [0]:
# Fetch the order payments table from bronze schema
order_payments_df = read_table("order_payments","bronze").filter(col("payment_value") > 0)

# Write the order_payments_df to the silver schema
write_table(df = order_payments_df, table_name = "order_payments", schema_name = "silver") 

##### Order Reviews

In [0]:
# Fetch the order reviews table from bronze schema
order_reviews_df = read_table("order_reviews","bronze")

# Filter the rows that includes at least one of review_score, review_comment_title or review_comment_message
order_reviews_df = order_reviews_df.filter(col("review_score").isNotNull() | col("review_comment_title").isNotNull() | col("review_comment_message").isNotNull())

# Handle the line breaks and carriage returns and add additional column to indicate if there are any comments
order_reviews_df = order_reviews_df.withColumn("review_comment_message", regexp_replace(col("review_comment_message"), r"[\n\r]", " ")) \
                                .withColumn("has_review_commnents", col("review_comment_message").isNotNull())

# Write the order_reviews_df to the silver schema
write_table(df = order_reviews_df, table_name = "order_reviews", schema_name = "silver")

##### Orders

In [0]:
# Fetch the orders table from bronze schema
orders_df = read_table("orders","bronze")

# Write the orders_df to the silver schema
write_table(df = orders_df, table_name = "orders", schema_name = "silver")

##### Product Category Name Translation

In [0]:
# Fetch the product category name translation table from bronze schema
product_category_name_translation_df = read_table("product_category_name_translation","bronze")

# rename columns and clean data
product_category_name_translation_df = product_category_name_translation_df \
                                            .withColumn("product_category_name_english", regexp_replace(col("product_category_name_english"),"_", " & ")) \
                                            .withColumnRenamed("product_category_name", "category_name_pt") \
                                            .withColumnRenamed("product_category_name_english", "category_name_eng")

##### Products

In [0]:
# Fetch product table from bronze schema
products_df = read_table("products","bronze")

# Broadcast Join Products and Product Category Name Translation table
products_df = products_df.join(broadcast(product_category_name_translation_df), products_df.product_category_name == product_category_name_translation_df.category_name_pt, "left")

# Clean category name and drop unnecessary columns
products_df = products_df \
    .withColumn("category_name_eng", when(col("category_name_eng").isNull(), "other").otherwise(col("category_name_eng"))) \
    .drop("category_name_pt", "product_category_name")

# Write the products_df to the silver schema
write_table(df = products_df, table_name = "products", schema_name = "silver")

##### Sellers

In [0]:
# Fetch sellers data from bronze schema
sellers_df = read_table("sellers","bronze")

# Add state names by joining with state mapping dataframe
sellers_df = sellers_df.join(broadcast(state_mapping_df), sellers_df.seller_state == state_mapping_df.state_code, how ="left").drop("state_code", "seller_state")

# Rename the state column to seller_state
sellers_df = sellers_df.withColumnRenamed("state", "seller_state")

# Write the sellers_df to the silver schema
write_table(df = sellers_df, table_name = "sellers", schema_name = "silver")